# C318 — Trabalho 2 — Notebook 03: AutoGluon oficial

Este notebook executa o AutoGluon Tabular 1.6.3 sobre a mesma base filtrada e a mesma partição 60/20/20 dos demais sistemas. Apenas o treino oficial é entregue ao `fit`; validação e teste permanecem externos.

No Google Colab, basta enviar este arquivo `.ipynb`: a primeira célula de ambiente baixa automaticamente os dados e módulos do projeto e instala as dependências. As execuções caras começam desligadas. Os predictors são preservados em `resultados/autogluon/modelos/` (ignorado pelo Git) e os artefatos auditáveis ficam em `resultados/autogluon/`.

## 0. Protocolo congelado antes dos resultados

- Curva: 5 minutos nas seeds 42, 101, 223, 347 e 409; 30 e 60 minutos somente na seed 42.
- Cada seed cria um holdout interno estratificado 80/20 usando exclusivamente o treino oficial.
- Configuração: `medium_quality`, F1-macro, CPU, oito classes e `label_count_threshold=1`.
- Seleção: maior média de F1-macro na validação externa; empate pelo menor budget.
- Ablação: 30 minutos/seed 42; o run com atributos é reaproveitado e apenas o run sem atributos é adicional.
- Campeão final: artefato da seed 42 no budget selecionado, sem refit e sem escolher a melhor seed.
- O teste é consultado uma única vez, depois de congelar a seleção.

In [ ]:
RUN_SMOKE = False
RUN_VALIDATION = False
RUN_DERIVED_ABLATION = False
RUN_AUDIT = False
RUN_FINAL_TEST = False

assert not (RUN_FINAL_TEST and RUN_VALIDATION), (
    "Não misture seleção de budget e consulta ao teste na mesma execução."
)

## 1. Ambiente, dados e diretórios

In [ ]:
# Bootstrap autossuficiente: no Colab, o usuário precisa enviar somente este notebook.
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

PROJECT_SOURCES = [
    ("https://github.com/PedroTiso/trabalho-2-topicos-especiais.git", "main"),
    ("https://github.com/vinigs22/trabalho-2-topicos-especiais.git",
     "feat/autogluon-oficial"),
]
COLAB_PROJECT_DIR = Path("/content/trabalho-2-topicos-especiais")
REQUIRED_AUTOGLOUON_VERSION = "1.6.3"

try:
    import google.colab  # type: ignore  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

def local_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "HPLC_data.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Fora do Colab, execute o notebook dentro do repositório clonado."
    )

def bootstrap_colab():
    markers = [
        COLAB_PROJECT_DIR / "data" / "HPLC_data.csv",
        COLAB_PROJECT_DIR / "data" / "splits" / "split_indices.csv",
        COLAB_PROJECT_DIR / "src" / "c318_common.py",
        COLAB_PROJECT_DIR / "src" / "autogluon_protocol.py",
        COLAB_PROJECT_DIR / "requirements-autogluon.txt",
    ]
    if not all(path.exists() for path in markers):
        if COLAB_PROJECT_DIR.exists() and any(COLAB_PROJECT_DIR.iterdir()):
            raise RuntimeError(
                f"Diretório incompleto em {COLAB_PROJECT_DIR}. "
                "Remova-o ou reinicie o runtime antes de tentar novamente."
            )
        print("Baixando dados e código auxiliar do projeto...")
        selected_source = None
        for repository, ref in PROJECT_SOURCES:
            raw_base = repository.removesuffix(".git").replace(
                "https://github.com/", "https://raw.githubusercontent.com/"
            )
            protocol_url = f"{raw_base}/{ref}/src/autogluon_protocol.py"
            try:
                with urlopen(protocol_url, timeout=15) as response:
                    if response.status == 200:
                        selected_source = (repository, ref)
                        break
            except (URLError, TimeoutError):
                continue
        if selected_source is None:
            raise RuntimeError(
                "Não foi possível localizar a versão oficial do protocolo no GitHub."
            )
        repository, ref = selected_source
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", ref,
             repository, str(COLAB_PROJECT_DIR)],
            check=True,
        )

    try:
        installed_version = version("autogluon.tabular")
    except PackageNotFoundError:
        installed_version = None
    if installed_version != REQUIRED_AUTOGLOUON_VERSION:
        print("Instalando AutoGluon e dependências; isso pode levar alguns minutos...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "--disable-pip-version-check", "-r",
             str(COLAB_PROJECT_DIR / "requirements-autogluon.txt")],
            check=True,
        )
    os.chdir(COLAB_PROJECT_DIR)
    return COLAB_PROJECT_DIR

ROOT = bootstrap_colab() if IN_COLAB else local_project_root()
print("Projeto pronto em:", ROOT)

In [ ]:
import json
import shutil
import subprocess
import sys
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

sys.path.insert(0, str(ROOT / "src"))

from c318_common import SEED, TARGET, compute_metrics, hardware_info, load_split
from autogluon_protocol import (
    AUTOGLOUON_FEATURES, BASE_FEATURES, OFFICIAL_RUNS, PRESET,
    append_unique_csv, fit_and_evaluate, index_hash, load_predictor,
    make_inner_holdout, package_versions, predictor_artifact_exists,
    predictions_frame, prepare_autogluon_data, run_id, select_budget, write_json,
)

RESULTS = ROOT / "resultados" / "autogluon"
MODELS = RESULTS / "modelos"
METADATA = RESULTS / "metadados"
PREDICTIONS = RESULTS / "predicoes_validacao"
LEADERBOARDS = RESULTS / "leaderboards"
LOGS = RESULTS / "logs"
FAILURES = RESULTS / "falhas_modelos"
for folder in [RESULTS, MODELS, METADATA, PREDICTIONS, LEADERBOARDS, LOGS, FAILURES]:
    folder.mkdir(parents=True, exist_ok=True)

AG_VERSION = version("autogluon.tabular")
HW = hardware_info()
VERSIONS = package_versions()
print("Raiz:", ROOT)
print("AutoGluon:", AG_VERSION)
print("Hardware:", HW)
print("Versões:", VERSIONS)

if any([RUN_SMOKE, RUN_VALIDATION, RUN_DERIVED_ABLATION, RUN_AUDIT, RUN_FINAL_TEST]):
    freeze = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        check=True, capture_output=True, text=True,
    ).stdout
    (RESULTS / "ambiente_pip_freeze.txt").write_text(freeze, encoding="utf-8")
    write_json(RESULTS / "ambiente.json", {"hardware": HW, "versoes": VERSIONS})

In [ ]:
train_df, val_df, test_df = load_split(ROOT)
train_ag = prepare_autogluon_data(train_df, with_derived=True)
val_ag = prepare_autogluon_data(val_df, with_derived=True)
# test_df é carregado para conferir o split, mas não é entregue ao predictor até RUN_FINAL_TEST.
LABELS = sorted(train_ag[TARGET].unique())

assert train_ag.shape == (7806, 18)
assert val_ag.shape == (2602, 18)
assert set(train_ag.index).isdisjoint(val_ag.index)
assert set(train_ag.index).isdisjoint(test_df.index)
assert set(val_ag.index).isdisjoint(test_df.index)
assert train_ag[TARGET].nunique() == val_ag[TARGET].nunique() == test_df[TARGET].nunique() == 8
print("Treino e validação preparados:", train_ag.shape, val_ag.shape)
display(train_ag[TARGET].value_counts().rename("treino").to_frame())

## 2. Persistência segura de cada run

In [ ]:
VALIDATION_CSV = RESULTS / "metricas_validacao_runs.csv"
ABLATION_RUNS_CSV = RESULTS / "metricas_ablacao_runs.csv"
SELECTION_JSON = RESULTS / "budget_selecionado.json"
SUMMARY_CSV = RESULTS / "resumo_budgets_validacao.csv"

def completed_ids(csv_path):
    if not csv_path.exists() or not csv_path.stat().st_size:
        return set()
    table = pd.read_csv(csv_path)
    return set(table["run_id"])

def save_predictor_audit_tables(predictor, current_run_id, model_path):
    leaderboard = predictor.leaderboard(extra_info=True, display=False)
    leaderboard.to_csv(LEADERBOARDS / f"{current_run_id}.csv", index=False)
    failures = predictor.model_failures(verbose=False)
    failures.to_csv(FAILURES / f"{current_run_id}.csv", index=False)
    try:
        from autogluon.tabular import TabularPredictor
        log_lines = TabularPredictor.load_log(predictor_path=str(model_path))
        (LOGS / f"{current_run_id}.txt").write_text("\n".join(line.rstrip() for line in log_lines), encoding="utf-8")
    except Exception as exc:
        (LOGS / f"{current_run_id}_erro.txt").write_text(
            f"{type(exc).__name__}: {exc}", encoding="utf-8"
        )
    return leaderboard, failures

def execute_run(budget, seed, with_derived, metrics_csv):
    current_run_id = run_id(budget, seed, with_derived=with_derived)
    model_path = MODELS / current_run_id
    already_recorded = current_run_id in completed_ids(metrics_csv)
    model_exists = predictor_artifact_exists(model_path)
    if already_recorded:
        if not model_exists:
            raise RuntimeError(
                f"{current_run_id} está no CSV, mas o predictor está ausente: {model_path}"
            )
        print("Já concluído:", current_run_id)
        return
    if model_path.exists():
        raise RuntimeError(
            f"Predictor parcial sem métrica: {model_path}. Audite antes de decidir descartá-lo."
        )

    source_train = prepare_autogluon_data(train_df, with_derived=with_derived)
    source_val = prepare_autogluon_data(val_df, with_derived=with_derived)
    metadata = {
        "run_id": current_run_id, "budget_min": budget, "seed": seed,
        "atributo_derivado": "com" if with_derived else "sem",
        "subset_fit": "train/holdout_interno", "subset_eval": "val",
        "preset": PRESET, "autogluon_version": AG_VERSION,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "hash_indices_treino_oficial": index_hash(source_train.index),
        "hash_indices_validacao_oficial": index_hash(source_val.index),
        "features": source_train.columns.drop(TARGET).tolist(),
    }
    try:
        predictor, metrics, pred, split_meta = fit_and_evaluate(
            source_train, source_val, model_path, budget, seed
        )
        pred.insert(0, "sl_no", val_df.loc[pred.index, "Sl No"].to_numpy())
        pred.to_csv(PREDICTIONS / f"{current_run_id}.csv", index_label="row_index")
        leaderboard, failures = save_predictor_audit_tables(
            predictor, current_run_id, model_path
        )
        row = {
            "sistema": "AutoGluon", "versao": AG_VERSION,
            "run_id": current_run_id, "budget_min": budget, "seed": seed,
            "atributo_derivado": "com" if with_derived else "sem",
            "subconjunto": "val", "modelo_vencedor_interno": predictor.model_best,
            **metrics,
        }
        append_unique_csv(metrics_csv, row)
        metadata.update({
            "status": "ok", "finished_at_utc": datetime.now(timezone.utc).isoformat(),
            "model_path": str(model_path.relative_to(ROOT)),
            "modelo_vencedor_interno": predictor.model_best,
            "classes_predictor": list(predictor.classes_),
            "numero_modelos": len(leaderboard), "numero_falhas_modelos": len(failures),
            "metricas": metrics, **split_meta,
        })
        print(current_run_id, "F1-macro externa:", f"{metrics['f1_macro']:.4f}")
    except Exception as exc:
        metadata.update({
            "status": "erro", "finished_at_utc": datetime.now(timezone.utc).isoformat(),
            "erro": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(),
        })
        raise
    finally:
        write_json(METADATA / f"{current_run_id}.json", metadata)

## 3. Smoke test sem validação externa nem teste

O smoke usa somente duas parcelas do treino oficial. Seus números não são resultados do trabalho.

In [ ]:
if RUN_SMOKE:
    smoke_path = MODELS / "smoke_60s_seed_42"
    if smoke_path.exists():
        raise RuntimeError(f"Smoke já existe: {smoke_path}")
    smoke_source, smoke_eval = make_inner_holdout(train_ag, SEED)
    smoke_predictor, smoke_metrics, _, _ = fit_and_evaluate(
        smoke_source, smoke_eval, smoke_path, budget_min=1, seed=SEED
    )
    reloaded = load_predictor(smoke_path)
    assert reloaded.model_best == smoke_predictor.model_best
    assert set(reloaded.classes_) == set(LABELS)
    print("Smoke concluído e recarregado. Métricas apenas diagnósticas:", smoke_metrics)
else:
    print("RUN_SMOKE=False. Nenhum treino foi iniciado.")

## 4. Sete execuções oficiais e seleção do budget

In [ ]:
if RUN_VALIDATION:
    for budget, seed in OFFICIAL_RUNS:
        execute_run(budget, seed, with_derived=True, metrics_csv=VALIDATION_CSV)
else:
    print("RUN_VALIDATION=False. Nenhuma busca oficial foi iniciada.")

validation_runs = (
    pd.read_csv(VALIDATION_CSV)
    if VALIDATION_CSV.exists() and VALIDATION_CSV.stat().st_size
    else pd.DataFrame()
)
if not validation_runs.empty:
    display(validation_runs)
    try:
        summary, chosen_budget = select_budget(validation_runs)
    except ValueError as exc:
        print("Seleção ainda não congelada:", exc)
    else:
        summary.to_csv(SUMMARY_CSV, index=False)
        write_json(SELECTION_JSON, {
            "criterio": "maior média de F1-macro na validação externa; empate pelo menor budget",
            "budget_selecionado_min": chosen_budget,
            "seed_campea": 42,
            "matriz_runs": [{"budget_min": b, "seed": s} for b, s in OFFICIAL_RUNS],
            "resumo": summary.to_dict(orient="records"),
        })
        display(summary)
        print("Budget congelado:", chosen_budget, "minutos; predictor final: seed 42")
else:
    print("Ainda não há runs oficiais.")

## 5. Ablação pareada em 30 minutos/seed 42

O resultado `com` vem da curva oficial. Somente `sem` dispara um novo treinamento.

In [ ]:
ABLATION_CSV = RESULTS / "ablacao_atributo_derivado.csv"
if RUN_DERIVED_ABLATION:
    if validation_runs.empty:
        raise RuntimeError("Execute primeiro a curva oficial.")
    with_row = validation_runs.query("budget_min == 30 and seed == 42")
    if len(with_row) != 1:
        raise RuntimeError("O run oficial de 30 minutos/seed 42 não está disponível.")
    execute_run(30, 42, with_derived=False, metrics_csv=ABLATION_RUNS_CSV)
    without_runs = pd.read_csv(ABLATION_RUNS_CSV)
    without_row = without_runs.query("budget_min == 30 and seed == 42")
    if len(without_row) != 1:
        raise RuntimeError("Run sem atributo derivado não foi registrado corretamente.")
    columns = [
        "atributo_derivado", "budget_min", "seed", "modelo_vencedor_interno",
        "f1_macro", "f1_weighted", "accuracy", "balanced_accuracy",
        "precision_macro", "recall_macro", "tempo_ajuste_s",
        "tempo_inferencia_s", "pico_memoria_mb",
    ]
    ablation = pd.concat([with_row, without_row], ignore_index=True)[columns]
    ablation.to_csv(ABLATION_CSV, index=False)
    display(ablation)
else:
    print("RUN_DERIVED_ABLATION=False. Nenhuma ablação foi iniciada.")

## 6. Auditoria e análise do modelo vencedor

A auditoria usa validação; não altera budget, seed ou `model_best`. O leaderboard externo é apenas diagnóstico posterior à seleção.

In [ ]:
AUDIT = RESULTS / "auditoria"
AUDIT.mkdir(parents=True, exist_ok=True)

if RUN_AUDIT:
    if not SELECTION_JSON.exists():
        raise FileNotFoundError("Congele primeiro o budget selecionado.")
    selection = json.loads(SELECTION_JSON.read_text(encoding="utf-8"))
    chosen_budget = int(selection["budget_selecionado_min"])
    champion_run_id = run_id(chosen_budget, 42, with_derived=True)
    champion_path = MODELS / champion_run_id
    champion = load_predictor(champion_path)
    winner = champion.model_best

    leaderboard_internal = champion.leaderboard(extra_info=True, display=False)
    leaderboard_external = champion.leaderboard(val_ag, extra_info=True, display=False)
    leaderboard_external = leaderboard_external.rename(columns={
        "score_test": "score_validacao_externa",
        "pred_time_test": "pred_time_validacao_externa",
        "pred_time_test_marginal": "pred_time_validacao_externa_marginal",
    })
    leaderboard_internal.to_csv(AUDIT / "leaderboard_interno.csv", index=False)
    leaderboard_external.to_csv(AUDIT / "leaderboard_validacao_externa.csv", index=False)
    write_json(AUDIT / "modelo_vencedor_info.json", champion.model_info(winner))
    write_json(AUDIT / "modelo_vencedor_hiperparametros.json",
               champion.model_hyperparameters(winner, output_format="all"))
    failures = champion.model_failures(verbose=False)
    failures.to_csv(AUDIT / "falhas_modelos.csv", index=False)
    write_json(AUDIT / "entrada_e_classes.json", {
        "run_id": champion_run_id, "modelo_vencedor": winner,
        "classes": list(champion.classes_),
        "atributos_originais": list(champion.original_features),
        "feature_metadata": champion.feature_metadata_in.to_dict(),
    })

    importance = champion.feature_importance(
        data=val_ag, model=winner, feature_stage="original",
        subsample_size=None, num_shuffle_sets=10, silent=False,
    ).sort_values("importance", ascending=False)
    importance.to_csv(AUDIT / "importancia_permutacao_validacao.csv", index_label="atributo")

    champion_predictions = pd.read_csv(
        PREDICTIONS / f"{champion_run_id}.csv", index_col="row_index"
    )
    report = classification_report(
        champion_predictions["y_true"], champion_predictions["y_pred"],
        labels=LABELS, output_dict=True, zero_division=0,
    )
    class_report = pd.DataFrame(report).T
    class_report.to_csv(AUDIT / "relatorio_por_classe_validacao.csv", index_label="classe")
    cm = confusion_matrix(
        champion_predictions["y_true"], champion_predictions["y_pred"], labels=LABELS
    )
    fig, ax = plt.subplots(figsize=(11, 9))
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(
        ax=ax, xticks_rotation=45, colorbar=False, values_format="d"
    )
    fig.tight_layout()
    fig.savefig(AUDIT / "matriz_confusao_validacao.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    graph_note = "Grafo não gerado (Graphviz/PyGraphviz indisponível)."
    try:
        graph_path = Path(champion.plot_ensemble_model(
            model=winner, prune_unused_nodes=True, filename="ensemble_model.png"
        ))
        shutil.copy2(graph_path, AUDIT / "ensemble_model.png")
        graph_note = "Grafo do ensemble salvo em ensemble_model.png."
    except Exception as exc:
        graph_note = f"Grafo não gerado: {type(exc).__name__}: {exc}"

    winner_internal = leaderboard_internal.loc[
        leaderboard_internal["model"] == winner
    ].iloc[0]
    winner_external = leaderboard_external.loc[
        leaderboard_external["model"] == winner
    ].iloc[0]
    rare = class_report[class_report.index.isin(LABELS)].sort_values("support").head(5)
    top_features = importance.head(10).reset_index().rename(columns={"index": "atributo"})
    ablation_text = "Ablação ainda não executada."
    if ABLATION_CSV.exists():
        ab = pd.read_csv(ABLATION_CSV).set_index("atributo_derivado")
        delta = ab.loc["com", "f1_macro"] - ab.loc["sem", "f1_macro"]
        ablation_text = f"Delta F1-macro (com - sem): {delta:+.6f}."
    analysis = f"""# Auditoria do AutoGluon vencedor

- Run congelado: `{champion_run_id}`; budget {chosen_budget} min; seed 42.
- Modelo interno vencedor: `{winner}` (stack level {winner_internal['stack_level']}).
- F1-macro interno (`score_val`): {winner_internal['score_val']:.6f}.
- F1-macro na validação externa: {winner_external['score_validacao_externa']:.6f}.
- Modelos treinados: {len(leaderboard_internal)}; falhas registradas: {len(failures)}.
- Atributos reconhecidos: {len(champion.original_features)}; classes preservadas: {len(champion.classes_)}.
- {ablation_text}
- {graph_note}

## Classes raras na validação

{rare.to_markdown()}

## Dez maiores importâncias por permutação

{top_features.to_markdown(index=False)}

## Interpretação

O `model_best` foi escolhido pelo AutoGluon usando apenas seu holdout interno. O score externo acima é diagnóstico posterior e não alterou o modelo, a seed ou o budget. Consulte `modelo_vencedor_info.json` e `modelo_vencedor_hiperparametros.json` para a composição e os parâmetros completos; `falhas_modelos.csv` distingue famílias não treinadas por erro.
"""
    (AUDIT / "auditoria_modelo_vencedor.md").write_text(analysis, encoding="utf-8")
    display(leaderboard_external.head(10))
    display(importance.head(10))
    print(analysis)
else:
    print("RUN_AUDIT=False. Nenhuma auditoria foi iniciada.")

## 7. Teste reservado — consulta única do campeão

Este bloco não produz leaderboard no teste e recusa uma segunda execução.

In [ ]:
TEST_CSV = RESULTS / "metricas_teste.csv"
if RUN_FINAL_TEST:
    if TEST_CSV.exists():
        raise RuntimeError(
            "O teste do AutoGluon já foi consultado. Use os artefatos salvos; não repita."
        )
    if not SELECTION_JSON.exists():
        raise FileNotFoundError("Não há seleção congelada.")
    selection = json.loads(SELECTION_JSON.read_text(encoding="utf-8"))
    chosen_budget = int(selection["budget_selecionado_min"])
    final_run_id = run_id(chosen_budget, 42, with_derived=True)
    final_predictor = load_predictor(MODELS / final_run_id)
    test_ag = prepare_autogluon_data(test_df, with_derived=True)
    test_predictions, test_metrics = predictions_frame(final_predictor, test_ag)
    test_row = {
        "sistema": "AutoGluon", "versao": AG_VERSION,
        "run_id": final_run_id, "budget_min": chosen_budget, "seed": 42,
        "subconjunto": "test",
        "modelo_vencedor_interno": final_predictor.model_best,
        **test_metrics,
    }
    pd.DataFrame([test_row]).to_csv(TEST_CSV, index=False)
    test_predictions.insert(
        0, "sl_no", test_df.loc[test_predictions.index, "Sl No"].to_numpy()
    )
    test_predictions.to_csv(RESULTS / "predicoes_teste.csv", index_label="row_index")
    write_json(RESULTS / "resumo_final.json", {
        **test_row, "consultado_em_utc": datetime.now(timezone.utc).isoformat(),
        "hash_indices_teste": index_hash(test_ag.index),
    })
    display(pd.DataFrame([test_row]))
else:
    print("RUN_FINAL_TEST=False. O conjunto de teste não foi consultado.")

## Ordem recomendada de execução

1. Envie somente este notebook ao Colab e execute tudo com as flags falsas. O bootstrap baixa dados/código e instala o AutoGluon automaticamente.
2. Execute `RUN_SMOKE=True` isoladamente.
3. Execute `RUN_VALIDATION=True` até completar os sete runs e congelar o budget.
4. Execute `RUN_DERIVED_ABLATION=True`.
5. Execute `RUN_AUDIT=True` e revise a análise.
6. Somente então execute `RUN_FINAL_TEST=True` uma vez.